In [1]:
%load_ext autoreload
%autoreload 2

from functools import partial

import moe
from tests import utils as test_utils

import jax
import jax.numpy as jnp
from jax import random
from jax.sharding import PartitionSpec as P, NamedSharding
import tune_jax
tune_jax.logger.setLevel("INFO")

# mesh experiments

In [2]:
n = jax.device_count()
mesh = jax.make_mesh((n,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.sharding.set_mesh(mesh)

In [3]:
fn = lambda: moe.utils.empty((1024, 1024), jnp.bfloat16, P(None, "x"))

In [4]:
fn_ = tune_jax.tune(fn)
fn_()

Compiling...:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling tpu:   0%|          | 0/5 [00:00<?, ?it/s]

Saving optimization profile to `/tmp/tuning_profile_2025-11-17_19:20:23_91fy8g4e`


Profiling tpu: 100%|██████████| 5/5 [00:04<00:00,  1.16it/s]


Array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=bfloat16)

# ra2a 2d

In [2]:
n_devices = jax.device_count()
mesh = jax.make_mesh((n_devices,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.set_mesh(mesh)

In [20]:
x_sort, input_offsets, send_sizes, output_offsets, recv_sizes = test_utils.generate_data(8 * 8 * 4096, 4096, n_devices, multiple=8)
x_sort = x_sort.reshape((x_sort.shape[0], -1)).astype(jnp.bfloat16)

In [29]:
@jax.jit
@partial(jax.shard_map, out_specs=P("x", None), check_vma=False)
def test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  #output = jnp.zeros((2 * x.shape[0],) + x.shape[1:], x.dtype)
  output = moe.utils.empty((2 * x.shape[0],) + x.shape[1:], x.dtype)
  with jax.named_scope("start"):
    out = moe.ra2a.ra2a_2d(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x", multiple=8)
  with jax.named_scope("jax.lax.ragged_all_to_all"):
    out2 = jax.lax.ragged_all_to_all(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
  return out, out2

In [34]:
out, out2 = test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes)
err = jnp.sum(jnp.abs(out - out2))
print(f"{err = }")

err = Array(0, dtype=bfloat16)


In [35]:
with jax.profiler.trace("/tmp/ra2a"):
  for _ in range(3):
    out, out2 = jax.block_until_ready(test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes))

# compute on

In [ ]:
import jax
import jax.numpy as jnp
from jax.experimental.compute_on import compute_on

import numpy as np

@jax.jit
@compute_on("tpu_sparsecore")
def my_gather(x, idx):
  return x[idx, ...]

x = jnp.ones((8192, 4096))
idx = jnp.argsort(np.random.randn(x.shape[0]))

In [6]:
with jax.profiler.trace("/tmp/compute_on"):
  for _ in range(3):
    jax.block_until_ready(my_gather(x, idx))